# 06_compare_models

Сводный ноутбук для сборки результатов после того, как baseline, SARIMA, tabular-модели и CatBoost сохранят свои метрики в `data/experiment_info/`.


In [ ]:
from pathlib import Path
import pandas as pd

EXP_INFO_DIR = Path("../data/experiment_info")

def load_metrics_if_exists(filename, model_name, scope=None):
    path = EXP_INFO_DIR / filename
    if not path.exists():
        return None
    df = pd.read_csv(path)
    if "model" not in df.columns:
        df["model"] = model_name
    if scope is not None and "scope" not in df.columns:
        df["scope"] = scope
    return df


In [ ]:
frames = [
    load_metrics_if_exists("mean_baseline_metrics_all_series.csv", "mean_baseline", "all_series"),
    load_metrics_if_exists("mean_baseline_metrics_top_pairs.csv", "mean_baseline", "top_pairs"),
    load_metrics_if_exists("sarima_store_metrics_by_split.csv", "sarima", "top_pairs"),
    load_metrics_if_exists("catboost_store_metrics_by_split_recursive.csv", "catboost", "top_pairs"),
]
metrics_df = pd.concat([df for df in frames if df is not None], ignore_index=True) if any(df is not None for df in frames) else pd.DataFrame()
metrics_df.head()


In [ ]:
if not metrics_df.empty:
    summary = metrics_df.groupby(["model", "scope"], as_index=False)[[c for c in ["RMSLE", "RMSE", "MAE", "WAPE"] if c in metrics_df.columns]].mean()
    summary = summary.sort_values(["scope", "RMSLE"], ascending=[True, True])
    summary
else:
    print("Метрики пока не собраны. Сначала выполните экспериментальные ноутбуки.")
